In [ ]:
import pandas as pd
import numpy as np
import unicodedata
from sklearn.preprocessing import MultiLabelBinarizer
from pathlib import Path
# ============================================
#  Função de normalização de strings e geração do dataset V4 - contendo dados de topografia
# ============================================

def remover_acentos(texto):
    if pd.isna(texto):
        return texto

    return "".join(
        c for c in unicodedata.normalize("NFKD", texto)
        if not unicodedata.combining(c)
    )

def troca_termo(termo, lista_termos_troca, feature):
    return [
        termo if categoria in lista_termos_troca else categoria
        for categoria in feature
    ]

# ============================================
#  Carregamento do banco de dados
# ============================================
caminho = "../exportacao_cavidade_QUADRILATERO_28092026.xlsx"
file_to_save = "df_treino_quadrilatero_v4"
variaveis_prospeccao = [
    "cavidade",
    "relevancia2009",
    "relevancia2017",
    "projecao_horizontal",
    "desnivel_topografia",
    "area",
    "volume",
    "zona_afotica",
    "raizes",
    "nome_entrada",
    "ocorrencia_guano",
    "tipo_cavidade",
    "dificuldades_internas",
    "litotipos_prospeccao",
    "tipos_espeleotema",
    "feicoes_internas",
    "feicoes_hidrologicas",
    "animais_observados",
    "arqueologia_paleontologia",

]
df = pd.read_excel(caminho, usecols=variaveis_prospeccao)


# ============================================
#  definição do target e remoção de desnivel inválido
# ============================================

df["target"] =( df["relevancia2017"]
               .fillna(df["relevancia2009"]))

df_treino = df[
    df["target"].notna()
    & (df["target"].astype(str).str.strip() != "")
].copy()


df_treino = df_treino.drop(index=df_treino[df_treino["target"]=="Estudo em andamento"].index)

df_treino["target"] = np.where(
    df_treino["target"] == "Máxima",
    1,
    0
)

print("+"*50)

print(f"Shape: {df.shape}")

print("+"*50)

df_treino = df_treino.drop(columns=["relevancia2017", "relevancia2009"]).dropna().copy()

mlb = MultiLabelBinarizer()

# ============================================
#  Padronizando animais_observados
# ============================================

animais_observados = (
    df_treino["animais_observados"]
    .str.lower()
    .apply(remover_acentos)
    .str.strip(";,")
    .str.split(r"\s*[;,]\s*")
)
animais_observados = animais_observados.apply(
    lambda lista:[
        "outros" if item == "outro" else item
        for item in lista
    ]
)

mlb.fit(animais_observados)

animais_encoded = pd.DataFrame(
    mlb.fit_transform(animais_observados),
    columns=[
    f"animal_observado_{classe}"
    for classe in mlb.classes_
],
    index=df_treino.index
)
df_treino = pd.concat(
    [
        df_treino.drop(columns=["animais_observados"]),
        animais_encoded
    ],
    axis=1

)
# ============================================
#  Padronizando tipos_espeleotema
# ============================================

tipos_espeleotema = df_treino["tipos_espeleotema"]
tipos_espeleotema = (tipos_espeleotema
                     .str.strip(",;")
                     .str.lower()
                     .apply(remover_acentos)
                     .str.split(r"\s*[;,]\s*")
                     )

tipos_espeleotema = tipos_espeleotema.apply(
    lambda lista: [
        "nenhum" if (item == "nenhuma" or item =="nao observado") else item
        for item in lista
    ]
)

mlb.fit(tipos_espeleotema)

tipos_espeleotema_encoded = pd.DataFrame(
    mlb.fit_transform(tipos_espeleotema),
    columns=[
        f"tipo_espeleotema_{classe}"
        for classe in mlb.classes_
    ],
    index = df_treino.index
)

df_treino = pd.concat(
    [
        df_treino.drop(columns=["tipos_espeleotema"]),
        tipos_espeleotema_encoded
    ],
    axis=1
)


# ============================================
#  Padronizando feicoes_hidrologicas
# ============================================

feicoes_hidrologicas = df_treino["feicoes_hidrologicas"]
feicoes_hidrologicas = (feicoes_hidrologicas
                     .str.strip(",;")
                     .str.lower()
                     .apply(remover_acentos)
                     .str.split(r"\s*[;,]\s*")
                     )

feicoes_hidrologicas.explode().value_counts()
feicoes_hidrologicas = feicoes_hidrologicas.apply(
    lambda lista: [
        "outro" if item == "outros" else item
        for item in lista
    ]
)
feicoes_hidrologicas = feicoes_hidrologicas.apply(
    lambda lista: [
        "nenhuma" if item in ["nenhum", "nao observado"] else item
        for item in lista
    ]
)
mlb.fit(feicoes_hidrologicas)

feicoes_hidrologicas_encoded = pd.DataFrame(
    mlb.fit_transform(feicoes_hidrologicas),
    columns=[
        f"feicao_hidrologica_{classe}"
        for classe in mlb.classes_
    ],
    index=df_treino.index
)

df_treino = pd.concat(
    [
        df_treino.drop(columns=["feicoes_hidrologicas"]),
        feicoes_hidrologicas_encoded
    ],
    axis=1
)

# ============================================
#  Padronizando dificuldades_internas
# ============================================

dificuldades_internas = df_treino["dificuldades_internas"]
dificuldades_internas = (dificuldades_internas
                     .str.strip(",;")
                     .str.lower()
                     .apply(remover_acentos)
                     .str.split(r"\s*[;,]\s*")
                     )


mlb.fit(dificuldades_internas)

dificuldades_internas_encoded = pd.DataFrame(
    mlb.fit_transform(dificuldades_internas),
    columns=[
        f"dificuldade_interna_{classe}"
        for classe in mlb.classes_
    ],
    index=df_treino.index
)

df_treino = pd.concat(
    [
        df_treino.drop(columns=["dificuldades_internas"]),
        dificuldades_internas_encoded
    ],
    axis=1
)

# ============================================
#  Padronizando feicoes_internas
# ============================================

feicoes_internas = df_treino["feicoes_internas"]
feicoes_internas = (feicoes_internas
                     .str.strip(",;")
                     .str.lower()
                     .apply(remover_acentos)
                     .str.split(r"\s*[;,]\s*")
                     )

feicoes_internas.explode().value_counts()

feicoes_internas = feicoes_internas.apply(
    lambda lista: [
        "outros" if item in ["outro"] else item
        for item in lista
    ]
)
feicoes_internas = feicoes_internas.apply(
    lambda lista: [
        "blocos abatidos" if item in ["blocosabatidos"] else item
        for item in lista
    ]
)
mlb.fit(feicoes_internas)

feicoes_internas_encoded = pd.DataFrame(
    mlb.fit_transform(feicoes_internas),
    columns=[f"feicao_interna_{classe}" for classe in mlb.classes_],
    index=df_treino.index
)

df_treino = pd.concat(
    [
        df_treino.drop(columns=["feicoes_internas"]),
        feicoes_internas_encoded
    ],
    axis=1
)

# ============================================
#  Padronizando arqueologia_paleontologia
# ============================================

arqueologia_paleontologia = df_treino["arqueologia_paleontologia"]
arqueologia_paleontologia = (arqueologia_paleontologia
                     .str.strip(",;")
                     .str.lower()
                     .apply(remover_acentos)
                     .str.split(r"\s*[;,]\s*")
                     )


arqueologia_paleontologia = arqueologia_paleontologia.apply(
    lambda lista: [
        "nenhum" if item in ["nao observado"] else item
        for item in lista
    ]
)

mlb.fit(arqueologia_paleontologia)

arqueologia_paleontologia_encoded = pd.DataFrame(
    mlb.fit_transform(arqueologia_paleontologia),
    columns=[
        f"arqueologia_paleontologia_{classe}"
        for classe in mlb.classes_
    ],
    index= df_treino.index
)

df_treino = pd.concat(
    [
        df_treino.drop(columns="arqueologia_paleontologia"),
        arqueologia_paleontologia_encoded
    ],
    axis = 1
)

# ============================================
#  Padronizando litotipos_prospeccao
# ============================================
litotipos_prospeccao = df_treino["litotipos_prospeccao"]
litotipos_prospeccao = (litotipos_prospeccao
                     .str.strip(",;")
                     .str.lower()
                     .apply(remover_acentos)
                     .str.split(r"\s*[;,]\s*")
                     )


litotipos_prospeccao = litotipos_prospeccao.apply(
    lambda lista: [
        "outros" if item in ["outra"] else item
        for item in lista
    ]
)

litotipos_prospeccao = litotipos_prospeccao.apply(
    lambda lista: [
        "outros" if item in ["outra"] else item
        for item in lista
    ]
)

litotipos_prospeccao = litotipos_prospeccao.apply(
    lambda lista: [
        "granito-gnaisse" if item in ["rochas granito–gnaissicas",
                                      "rochas granito – gnaissicas" ] else item
        for item in lista
    ]
)

litotipos_prospeccao = litotipos_prospeccao.apply(
    lambda lista: [
        "rochas siliciosas" if item in ["rochas silicosas"] else item
        for item in lista
    ]
)

mlb.fit(litotipos_prospeccao)
litotipos_prospeccao_encoded = pd.DataFrame(
    mlb.fit_transform(litotipos_prospeccao),
    columns=[
        f"litotipo_prospeccao_{classe}"
        for classe in mlb.classes_
    ],
    index=df_treino.index
)

df_treino = pd.concat(
    [
        df_treino.drop(columns=["litotipos_prospeccao"]),
        litotipos_prospeccao_encoded
    ],
    axis=1
)
df_treino

# ============================================
#  Padronizando nome_entrada
# ============================================

df_treino["nome_entrada"] = (df_treino["nome_entrada"]
                     .str.strip()
                     .str.lower()
                     .apply(remover_acentos)
                     .replace(
                         {
                             "u": "1",
                             "unico": "1",
                             "principal": "1",
                             "unica": "1"
                         }
                     )
                     .str.replace(",", ".", regex=False)
                     .pipe(pd.to_numeric)
                     )


# ============================================
#  Padronizando ocorrencia_guano
# ============================================
ocorrencia_guano = df_treino["ocorrencia_guano"]
ocorrencia_guano = (ocorrencia_guano
                     .str.strip(",;")
                     .str.lower()
                     .apply(remover_acentos)
                     .str.split(r"\s*[;,]\s*")
                     )

ocorrencia_guano.explode().value_counts()
ocorrencia_guano = ocorrencia_guano.apply(
    lambda lista: [
        "nao" if item in ["nao observado" ] else item
        for item in lista
    ]
)

ocorrencia_guano = ocorrencia_guano.apply(
    lambda lista: [
        "concentrado" if item in ["concetrado" ] else item
        for item in lista
    ]
)

mlb.fit(ocorrencia_guano)
ocorrencia_guano_encoded = pd.DataFrame(
    mlb.fit_transform(ocorrencia_guano),
    columns=[
        f"ocorrencia_guano_{classe}"
        for classe in mlb.classes_
    ],
    index=df_treino.index
)

df_treino = pd.concat(
    [
        df_treino.drop(columns=["ocorrencia_guano"]),
        ocorrencia_guano_encoded
    ],
    axis=1
)

# ============================================
#  Padronizando Raizes e zona_afotica
# ============================================

df_treino["zona_afotica"] = (df_treino["zona_afotica"]
                             .str.strip()
                             .str.lower()
                             .apply(remover_acentos))

df_treino["raizes"] = (df_treino["raizes"]
                             .str.strip()
                             .str.lower()
                             .apply(remover_acentos))



df_treino["zona_afotica"] = np.where(
    df_treino["zona_afotica"] == "sim",
    1,
    0    
)
df_treino["raizes"] = np.where(
    df_treino["raizes"] == "sim",
    1,
    0    
)
# ============================================
#  Padronizando Raizes e tipo_cavidade
# ============================================

df_treino["tipo_cavidade"] = (df_treino["tipo_cavidade"]
                             .str.strip()
                             .str.lower()
                             .apply(remover_acentos)
                             )

tipo_encoded = pd.get_dummies(
    df_treino["tipo_cavidade"],
    prefix="tipo_cavidade_",
    dtype=int
)
df_treino= pd.concat(
    [df_treino.drop(columns=["tipo_cavidade"]),
     tipo_encoded],
     axis=1
)

columns_to_drop = [
    "animal_observado_outros",
    "tipo_espeleotema_nenhum",
    "tipo_espeleotema_outros",
    "feicao_hidrologica_nenhuma",
    "feicao_hidrologica_outro",
    "dificuldade_interna_nenhuma",
    "dificuldade_interna_outros",
    "dificuldade_interna_natacao",
    "dificuldade_interna_cachoeira",
    "dificuldade_interna_sifao",
    "feicao_interna_nenhum",
    "feicao_interna_outros",
    "feicao_interna_nao observado",
    "arqueologia_paleontologia_arte rupestre",
    "arqueologia_paleontologia_fossil",
    "arqueologia_paleontologia_nenhum",
    "arqueologia_paleontologia_outros",
    "litotipo_prospeccao_outros",
    "ocorrencia_guano_sim",
    "ocorrencia_guano_nao",
    "tipo_cavidade__galeria"
]

df_treino = df_treino.drop(columns=columns_to_drop, errors="ignore").copy()

df_treino.columns = (
    df_treino.columns
    .str.replace(" ", "_", regex=False)
    .str.replace("-","_", regex=False)
)

output_dir = Path('../datasets')
output_dir.mkdir(parents=True, exist_ok=True)

df_treino.to_csv(output_dir / f"{file_to_save}.csv", index=False)
df_treino.to_excel(output_dir / f"{file_to_save}.xlsx", index=False)



In [ ]:
import pandas as pd
import numpy as np
import unicodedata
from sklearn.preprocessing import MultiLabelBinarizer
from pathlib import Path
# ============================================
#  Função de normalização de strings e geração do dataset V3 - contendo apenas dados da prospecção
# ============================================

def remover_acentos(texto):
    if pd.isna(texto):
        return texto

    return "".join(
        c for c in unicodedata.normalize("NFKD", texto)
        if not unicodedata.combining(c)
    )

def troca_termo(termo, lista_termos_troca, feature):
    return [
        termo if categoria in lista_termos_troca else categoria
        for categoria in feature
    ]

# ============================================
#  Carregamento do banco de dados
# ============================================
caminho = "../exportacao_cavidade_QUADRILATERO_28092026.xlsx"
file_to_save = "df_treino_quadrilatero_v3"
variaveis_prospeccao = [
    "cavidade",
    "relevancia2009",
    "relevancia2017",
    "desenv_aproximado",
    "desnivel_propeccao",
    "zona_afotica",
    "raizes",
    "nome_entrada",
    "ocorrencia_guano",
    "tipo_cavidade",
    "dificuldades_internas",
    "litotipos_prospeccao",
    "tipos_espeleotema",
    "feicoes_internas",
    "feicoes_hidrologicas",
    "animais_observados",
    "arqueologia_paleontologia",

]
df = pd.read_excel(caminho, usecols=variaveis_prospeccao)


# ============================================
#  definição do target e remoção de desnivel inválido
# ============================================

df["target"] =( df["relevancia2017"]
               .fillna(df["relevancia2009"]))

df_treino = df[
    df["target"].notna()
    & (df["target"].astype(str).str.strip() != "")
].copy()

df_treino = df_treino.drop(index=df_treino[df_treino["target"]=="Estudo em andamento"].index)

df_treino["target"] = np.where(
    df_treino["target"] == "Máxima",
    1,
    0
)

print("+"*50)

print(f"Shape: {df.shape}")

print("+"*50)

df_treino = df_treino.drop(columns=["relevancia2017", "relevancia2009"]).dropna().copy()

mlb = MultiLabelBinarizer()

# ============================================
#  Padronizando animais_observados
# ============================================

animais_observados = (
    df_treino["animais_observados"]
    .str.lower()
    .apply(remover_acentos)
    .str.strip(";,")
    .str.split(r"\s*[;,]\s*")
)
animais_observados = animais_observados.apply(
    lambda lista:[
        "outros" if item == "outro" else item
        for item in lista
    ]
)

mlb.fit(animais_observados)

animais_encoded = pd.DataFrame(
    mlb.fit_transform(animais_observados),
    columns=[
    f"animal_observado_{classe}"
    for classe in mlb.classes_
],
    index=df_treino.index
)
df_treino = pd.concat(
    [
        df_treino.drop(columns=["animais_observados"]),
        animais_encoded
    ],
    axis=1

)
# ============================================
#  Padronizando tipos_espeleotema
# ============================================

tipos_espeleotema = df_treino["tipos_espeleotema"]
tipos_espeleotema = (tipos_espeleotema
                     .str.strip(",;")
                     .str.lower()
                     .apply(remover_acentos)
                     .str.split(r"\s*[;,]\s*")
                     )

tipos_espeleotema = tipos_espeleotema.apply(
    lambda lista: [
        "nenhum" if (item == "nenhuma" or item =="nao observado") else item
        for item in lista
    ]
)

mlb.fit(tipos_espeleotema)

tipos_espeleotema_encoded = pd.DataFrame(
    mlb.fit_transform(tipos_espeleotema),
    columns=[
        f"tipo_espeleotema_{classe}"
        for classe in mlb.classes_
    ],
    index = df_treino.index
)

df_treino = pd.concat(
    [
        df_treino.drop(columns=["tipos_espeleotema"]),
        tipos_espeleotema_encoded
    ],
    axis=1
)


# ============================================
#  Padronizando feicoes_hidrologicas
# ============================================

feicoes_hidrologicas = df_treino["feicoes_hidrologicas"]
feicoes_hidrologicas = (feicoes_hidrologicas
                     .str.strip(",;")
                     .str.lower()
                     .apply(remover_acentos)
                     .str.split(r"\s*[;,]\s*")
                     )

feicoes_hidrologicas.explode().value_counts()
feicoes_hidrologicas = feicoes_hidrologicas.apply(
    lambda lista: [
        "outro" if item == "outros" else item
        for item in lista
    ]
)
feicoes_hidrologicas = feicoes_hidrologicas.apply(
    lambda lista: [
        "nenhuma" if item in ["nenhum", "nao observado"] else item
        for item in lista
    ]
)
mlb.fit(feicoes_hidrologicas)

feicoes_hidrologicas_encoded = pd.DataFrame(
    mlb.fit_transform(feicoes_hidrologicas),
    columns=[
        f"feicao_hidrologica_{classe}"
        for classe in mlb.classes_
    ],
    index=df_treino.index
)

df_treino = pd.concat(
    [
        df_treino.drop(columns=["feicoes_hidrologicas"]),
        feicoes_hidrologicas_encoded
    ],
    axis=1
)

# ============================================
#  Padronizando dificuldades_internas
# ============================================

dificuldades_internas = df_treino["dificuldades_internas"]
dificuldades_internas = (dificuldades_internas
                     .str.strip(",;")
                     .str.lower()
                     .apply(remover_acentos)
                     .str.split(r"\s*[;,]\s*")
                     )


mlb.fit(dificuldades_internas)

dificuldades_internas_encoded = pd.DataFrame(
    mlb.fit_transform(dificuldades_internas),
    columns=[
        f"dificuldade_interna_{classe}"
        for classe in mlb.classes_
    ],
    index=df_treino.index
)

df_treino = pd.concat(
    [
        df_treino.drop(columns=["dificuldades_internas"]),
        dificuldades_internas_encoded
    ],
    axis=1
)

# ============================================
#  Padronizando feicoes_internas
# ============================================

feicoes_internas = df_treino["feicoes_internas"]
feicoes_internas = (feicoes_internas
                     .str.strip(",;")
                     .str.lower()
                     .apply(remover_acentos)
                     .str.split(r"\s*[;,]\s*")
                     )

feicoes_internas.explode().value_counts()

feicoes_internas = feicoes_internas.apply(
    lambda lista: [
        "outros" if item in ["outro"] else item
        for item in lista
    ]
)
feicoes_internas = feicoes_internas.apply(
    lambda lista: [
        "blocos abatidos" if item in ["blocosabatidos"] else item
        for item in lista
    ]
)
mlb.fit(feicoes_internas)

feicoes_internas_encoded = pd.DataFrame(
    mlb.fit_transform(feicoes_internas),
    columns=[f"feicao_interna_{classe}" for classe in mlb.classes_],
    index=df_treino.index
)

df_treino = pd.concat(
    [
        df_treino.drop(columns=["feicoes_internas"]),
        feicoes_internas_encoded
    ],
    axis=1
)

# ============================================
#  Padronizando arqueologia_paleontologia
# ============================================

arqueologia_paleontologia = df_treino["arqueologia_paleontologia"]
arqueologia_paleontologia = (arqueologia_paleontologia
                     .str.strip(",;")
                     .str.lower()
                     .apply(remover_acentos)
                     .str.split(r"\s*[;,]\s*")
                     )


arqueologia_paleontologia = arqueologia_paleontologia.apply(
    lambda lista: [
        "nenhum" if item in ["nao observado"] else item
        for item in lista
    ]
)

mlb.fit(arqueologia_paleontologia)

arqueologia_paleontologia_encoded = pd.DataFrame(
    mlb.fit_transform(arqueologia_paleontologia),
    columns=[
        f"arqueologia_paleontologia_{classe}"
        for classe in mlb.classes_
    ],
    index= df_treino.index
)

df_treino = pd.concat(
    [
        df_treino.drop(columns="arqueologia_paleontologia"),
        arqueologia_paleontologia_encoded
    ],
    axis = 1
)

# ============================================
#  Padronizando litotipos_prospeccao
# ============================================
litotipos_prospeccao = df_treino["litotipos_prospeccao"]
litotipos_prospeccao = (litotipos_prospeccao
                     .str.strip(",;")
                     .str.lower()
                     .apply(remover_acentos)
                     .str.split(r"\s*[;,]\s*")
                     )


litotipos_prospeccao = litotipos_prospeccao.apply(
    lambda lista: [
        "outros" if item in ["outra"] else item
        for item in lista
    ]
)

litotipos_prospeccao = litotipos_prospeccao.apply(
    lambda lista: [
        "outros" if item in ["outra"] else item
        for item in lista
    ]
)

litotipos_prospeccao = litotipos_prospeccao.apply(
    lambda lista: [
        "granito-gnaisse" if item in ["rochas granito–gnaissicas",
                                      "rochas granito – gnaissicas" ] else item
        for item in lista
    ]
)

litotipos_prospeccao = litotipos_prospeccao.apply(
    lambda lista: [
        "rochas siliciosas" if item in ["rochas silicosas"] else item
        for item in lista
    ]
)

mlb.fit(litotipos_prospeccao)
litotipos_prospeccao_encoded = pd.DataFrame(
    mlb.fit_transform(litotipos_prospeccao),
    columns=[
        f"litotipo_prospeccao_{classe}"
        for classe in mlb.classes_
    ],
    index=df_treino.index
)

df_treino = pd.concat(
    [
        df_treino.drop(columns=["litotipos_prospeccao"]),
        litotipos_prospeccao_encoded
    ],
    axis=1
)
df_treino

# ============================================
#  Padronizando nome_entrada
# ============================================

df_treino["nome_entrada"] = (df_treino["nome_entrada"]
                     .str.strip()
                     .str.lower()
                     .apply(remover_acentos)
                     .replace(
                         {
                             "u": "1",
                             "unico": "1",
                             "principal": "1",
                             "unica": "1"
                         }
                     )
                     .str.replace(",", ".", regex=False)
                     .pipe(pd.to_numeric)
                     )


# ============================================
#  Padronizando ocorrencia_guano
# ============================================
ocorrencia_guano = df_treino["ocorrencia_guano"]
ocorrencia_guano = (ocorrencia_guano
                     .str.strip(",;")
                     .str.lower()
                     .apply(remover_acentos)
                     .str.split(r"\s*[;,]\s*")
                     )

ocorrencia_guano.explode().value_counts()
ocorrencia_guano = ocorrencia_guano.apply(
    lambda lista: [
        "nao" if item in ["nao observado" ] else item
        for item in lista
    ]
)

ocorrencia_guano = ocorrencia_guano.apply(
    lambda lista: [
        "concentrado" if item in ["concetrado" ] else item
        for item in lista
    ]
)

mlb.fit(ocorrencia_guano)
ocorrencia_guano_encoded = pd.DataFrame(
    mlb.fit_transform(ocorrencia_guano),
    columns=[
        f"ocorrencia_guano_{classe}"
        for classe in mlb.classes_
    ],
    index=df_treino.index
)

df_treino = pd.concat(
    [
        df_treino.drop(columns=["ocorrencia_guano"]),
        ocorrencia_guano_encoded
    ],
    axis=1
)

# ============================================
#  Padronizando Raizes e zona_afotica
# ============================================

df_treino["zona_afotica"] = (df_treino["zona_afotica"]
                             .str.strip()
                             .str.lower()
                             .apply(remover_acentos))

df_treino["raizes"] = (df_treino["raizes"]
                             .str.strip()
                             .str.lower()
                             .apply(remover_acentos))



df_treino["zona_afotica"] = np.where(
    df_treino["zona_afotica"] == "sim",
    1,
    0    
)
df_treino["raizes"] = np.where(
    df_treino["raizes"] == "sim",
    1,
    0    
)
# ============================================
#  Padronizando Raizes e tipo_cavidade
# ============================================

df_treino["tipo_cavidade"] = (df_treino["tipo_cavidade"]
                             .str.strip()
                             .str.lower()
                             .apply(remover_acentos)
                             )

tipo_encoded = pd.get_dummies(
    df_treino["tipo_cavidade"],
    prefix="tipo_cavidade_",
    dtype=int
)
df_treino= pd.concat(
    [df_treino.drop(columns=["tipo_cavidade"]),
     tipo_encoded],
     axis=1
)

columns_to_drop = [
    "animal_observado_outros",
    "tipo_espeleotema_nenhum",
    "tipo_espeleotema_outros",
    "feicao_hidrologica_nenhuma",
    "feicao_hidrologica_outro",
    "dificuldade_interna_nenhuma",
    "dificuldade_interna_outros",
    "dificuldade_interna_natacao",
    "dificuldade_interna_cachoeira",
    "dificuldade_interna_sifao",
    "feicao_interna_nenhum",
    "feicao_interna_outros",
    "feicao_interna_nao observado",
    "arqueologia_paleontologia_arte rupestre",
    "arqueologia_paleontologia_fossil",
    "arqueologia_paleontologia_nenhum",
    "arqueologia_paleontologia_outros",
    "litotipo_prospeccao_outros",
    "ocorrencia_guano_sim",
    "ocorrencia_guano_nao",
    "tipo_cavidade__galeria"
]

df_treino = df_treino.drop(columns=columns_to_drop, errors="ignore").copy()

df_treino.columns= (
    df_treino.columns
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
    )

output_dir = Path("../datasets")
output_dir.mkdir(exist_ok=True, parents=True)

df_treino.to_csv(output_dir / f"{file_to_save}.csv", index=False)
df_treino.to_excel(output_dir / f"{file_to_save}.xlsx", index=False)

In [ ]:
import pandas as pd
from pathlib import Path
output_dir = Path("../datasets")
df = pd.read_excel(output_dir / "df_treino_v3.xlsx")


In [ ]:
df

In [ ]:
teste = df[df["cavidade"].str.contains("0046")]
teste = teste.columns[df.loc[0] == 1]
teste


In [ ]:
import pandas as pd
import numpy as np
import unicodedata
from sklearn.preprocessing import MultiLabelBinarizer
from pathlib import Path
# ============================================
#  Função de normalização de strings e geração do dataset V3 - contendo apenas dados da prospecção
# ============================================

def remover_acentos(texto):
    if pd.isna(texto):
        return texto

    return "".join(
        c for c in unicodedata.normalize("NFKD", texto)
        if not unicodedata.combining(c)
    )

def troca_termo(termo, lista_termos_troca, feature):
    return [
        termo if categoria in lista_termos_troca else categoria
        for categoria in feature
    ]

# ============================================
#  Carregamento do banco de dados
# ============================================
caminho = "../exportacao_cavidade_QUADRILATERO_28092026.xlsx"
file_to_save = "df_treino_quadrilatero_v3"
variaveis_prospeccao = [
    "cavidade",
    "relevancia2009",
    "relevancia2017",
    "desenv_aproximado",
    "desnivel_propeccao",
    "zona_afotica",
    "raizes",
    "nome_entrada",
    "ocorrencia_guano",
    "tipo_cavidade",
    "dificuldades_internas",
    "litotipos_prospeccao",
    "tipos_espeleotema",
    "feicoes_internas",
    "feicoes_hidrologicas",
    "animais_observados",
    "arqueologia_paleontologia",

]
df = pd.read_excel(caminho, usecols=variaveis_prospeccao)


# ============================================
#  definição do target e remoção de desnivel inválido
# ============================================

df["target"] =( df["relevancia2017"]
               .fillna(df["relevancia2009"]))


df